In [0]:
from pyspark.sql import functions as F
import uuid

CATALOG     = "olist"
AUDIT_TABLE = f"{CATALOG}.gold.dq_audit"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {AUDIT_TABLE} (
    run_id     STRING,
    layer      STRING,
    table_name STRING,
    check_name STRING,
    severity   STRING,
    passed     BOOLEAN,
    observed   STRING,
    expected   STRING,
    checked_at TIMESTAMP
)
""")

RUN_ID   = str(uuid.uuid4())
_results = []

def check(layer, table, name, passed, observed, expected, severity="critical"):
    passed = bool(passed)
    _results.append((RUN_ID, layer, table, name, severity,
                     passed, str(observed), str(expected)))
    print(f"[{'PASS' if passed else 'FAIL'}] {table}.{name}  "
          f"observed={observed}  expected={expected}")

def flush():
    if not _results:
        print("no checks to write")
        return
    schema = ("run_id string, layer string, table_name string, check_name string, "
              "severity string, passed boolean, observed string, expected string")
    (spark.createDataFrame(_results, schema)
        .withColumn("checked_at", F.current_timestamp())
        .write.mode("append").saveAsTable(AUDIT_TABLE))
    print(f"wrote {len(_results)} results to {AUDIT_TABLE}")

print("ready, run_id =", RUN_ID)

In [0]:
orders = spark.table(f"{CATALOG}.silver.orders")
items  = spark.table(f"{CATALOG}.silver.order_items")

total  = orders.count()
unique = orders.select("order_id").distinct().count()
check("silver", "orders", "pk_unique", total == unique, unique, total)

nulls = orders.filter(F.col("order_id").isNull()).count()
check("silver", "orders", "pk_not_null", nulls == 0, nulls, 0)

orphans = items.join(orders.select("order_id"), "order_id", "left_anti").count()
check("silver", "order_items", "fk_order_exists", orphans == 0, orphans, 0)

impossible = orders.filter(F.col("delivered_at") < F.col("purchased_at")).count()
check("silver", "orders", "delivered_after_purchase", impossible == 0, impossible, 0)

KNOWN = {"delivered", "shipped", "canceled", "invoiced",
         "processing", "unavailable", "created", "approved"}
found   = {r["order_status"] for r in orders.select("order_status").distinct().collect()}
unknown = found - KNOWN
check("silver", "orders", "status_in_domain",
      len(unknown) == 0, sorted(unknown) or "none", "known set")

missing_ts = orders.filter(
    (F.col("order_status") == "delivered") & F.col("delivered_at").isNull()).count()
check("silver", "orders", "delivered_has_timestamp",
      missing_ts < total * 0.001, missing_ts, "< 0.1%", severity="warn")

# drift — safe on first run, the table exists but may be empty
prev = spark.sql(f"""
    SELECT observed FROM {AUDIT_TABLE}
    WHERE check_name = 'row_count' AND table_name = 'orders'
    ORDER BY checked_at DESC LIMIT 1
""").collect()

if prev:
    before = int(prev[0][0])
    drift  = abs(total - before) / before if before else 0
    check("silver", "orders", "volume_drift", drift < 0.20,
          f"{drift:.1%}", "< 20%", severity="warn")
else:
    print("[SKIP] volume_drift — no prior run to compare against")

check("silver", "orders", "row_count", True, total, "n/a", severity="info")

flush()

failed = [r for r in _results if not r[5] and r[4] == "critical"]
if failed:
    raise Exception(f"{len(failed)} critical check(s) failed: {[r[3] for r in failed]}")
print("all critical checks passed")

In [0]:
%sql
DESCRIBE HISTORY olist.silver.orders;

In [0]:
%sql
SELECT COUNT(*) FROM olist.silver.orders VERSION AS OF 0;

In [0]:
%sql
SELECT COUNT(*) FROM olist.silver.orders VERSION AS OF 0;

In [0]:
%sql
-- harmless no-op write that creates version 1
INSERT INTO olist.silver.orders SELECT * FROM olist.silver.orders WHERE 1=0;

In [0]:
%sql
DESCRIBE HISTORY olist.silver.orders;

In [0]:
%sql
SELECT 0 AS v, COUNT(*) AS rows FROM olist.silver.orders VERSION AS OF 0
UNION ALL
SELECT 1, COUNT(*) FROM olist.silver.orders VERSION AS OF 1;